# Przygotowanie datasetu
Uruchom z katalogu `garbage-yolo`. Powstaje `dataset_clean/`; oryginał pozostaje bez zmian.
Zachowujemy 6 klas i podział train/valid/test. Wielokąty YOLO zamieniamy na prostokąty.
Odrzucamy uszkodzone zdjęcia, brakujące/puste/błędne etykiety (całą parę, aby nie zostawiać nieoznaczonych obiektów), osierocone etykiety i zbędne pliki/cache.
Puste etykiety pomijamy, bo nie mamy potwierdzenia, że to zdjęcia samego tła.
Usuwamy identyczne pliki obrazów także między splitami (pierwszeństwo: test, valid, train); podobne zdjęcia wymagają osobnego przeglądu.

In [ ]:
%pip install -q "Pillow>=10,<13" "PyYAML>=6,<7"

In [ ]:
from pathlib import Path
from collections import Counter
from PIL import Image
import hashlib, json, math, shutil, yaml

ROOT = Path.cwd()
SRC = ROOT / "Trash Detection Dataset/CUSTOM_DATASET"
OUT = ROOT / "dataset_clean"
EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
config = yaml.safe_load((SRC / "data.yaml").read_text())
names = config["names"]
names = dict(enumerate(names)) if isinstance(names, list) else {int(k): v for k, v in names.items()}
assert sorted(names) == list(range(len(names))), "ID klas muszą być ciągłe od 0."
for split in ("train", "valid", "test"):
    assert (SRC / split / "images").is_dir() and (SRC / split / "labels").is_dir()
assert not OUT.exists(), "Folder dataset_clean już istnieje. Ustaw nowy OUT, aby przygotować dane ponownie."

In [ ]:
def to_box(line):
    values = list(map(float, line.split()))
    if not values or not all(map(math.isfinite, values)):
        raise ValueError("Puste lub niefinitywne wartości")
    cls, *coords = values
    if not cls.is_integer() or int(cls) not in names:
        raise ValueError("Nieprawidłowa klasa")
    polygon = len(coords) >= 6 and len(coords) % 2 == 0
    if polygon:
        if not all(0 <= v <= 1 for v in coords):
            raise ValueError("Wielokąt poza obrazem")
        xs, ys = coords[::2], coords[1::2]
        x1, y1, x2, y2 = min(xs), min(ys), max(xs), max(ys)
    elif len(coords) == 4:
        x, y, w, h = coords
        if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
            raise ValueError("Nieprawidłowy box")
        x1, y1, x2, y2 = x-w/2, y-h/2, x+w/2, y+h/2
    else:
        raise ValueError("Nieprawidłowy format YOLO")
    if min(x1, y1) < -1e-6 or max(x2, y2) > 1+1e-6 or x2 <= x1 or y2 <= y1:
        raise ValueError("Box poza obrazem lub bez powierzchni")
    x1, y1, x2, y2 = [min(1., max(0., v)) for v in (x1, y1, x2, y2)]
    box = ((x1+x2)/2, (y1+y2)/2, x2-x1, y2-y1)
    return f"{int(cls)} " + " ".join(f"{v:.12g}" for v in box), polygon

In [ ]:
OUT.mkdir(parents=True)
seen, report, skipped = set(), {}, []
for split in ("test", "valid", "train"):
    images, labels = SRC / split / "images", SRC / split / "labels"
    dst = OUT / split
    (dst / "images").mkdir(parents=True)
    (dst / "labels").mkdir()
    paths = sorted(p for p in images.iterdir() if p.is_file() and p.suffix.lower() in EXT)
    stems = Counter(p.stem for p in paths)
    stats = Counter(input_images=len(paths))
    stats["orphan_labels"] = sum(p.stem not in stems for p in labels.glob("*.txt"))
    for image in paths:
        label = labels / (image.stem + ".txt")
        try:
            if stems[image.stem] != 1:
                raise ValueError("Niejednoznaczna nazwa obrazu")
            with Image.open(image) as im:
                im.verify()
            with Image.open(image) as im:
                im.load()
                if min(im.size) < 10:
                    raise ValueError("Obraz mniejszy niż 10 px")
            lines = [s for s in label.read_text(encoding="utf-8").splitlines() if s.strip()]
            if not lines:
                raise ValueError("Pusta etykieta")
            converted = [to_box(s) for s in lines]
        except (OSError, ValueError, SyntaxError) as error:
            stats["invalid_pairs"] += 1
            skipped.append({"split": split, "file": image.name, "reason": str(error)})
            continue
        digest = hashlib.sha256(image.read_bytes()).hexdigest()
        if digest in seen:
            stats["duplicate_images"] += 1
            skipped.append({"split": split, "file": image.name, "reason": "Identyczny plik obrazu"})
            continue
        seen.add(digest)
        boxes = list(dict.fromkeys(box for box, _ in converted))
        shutil.copy2(image, dst / "images" / image.name)
        (dst / "labels" / label.name).write_text("\n".join(boxes) + "\n", encoding="utf-8")
        stats.update(kept_images=1, boxes=len(boxes), polygons_converted=sum(p for _, p in converted),
                     duplicate_boxes=len(converted)-len(boxes))
    report[split] = dict(stats)

(OUT / "prep_report.json").write_text(json.dumps({"summary": report, "skipped": skipped}, indent=2, ensure_ascii=False))
assert all(report[s].get("kept_images", 0) for s in report), "Co najmniej jeden split jest pusty — sprawdź raport."
(OUT / "data.yaml").write_text(yaml.safe_dump({
    "path": str(OUT.resolve()), "train": "train/images", "val": "valid/images", "test": "test/images",
    "nc": len(names), "names": names,
}, sort_keys=False, allow_unicode=True), encoding="utf-8")
print(json.dumps(report, indent=2))
print("Gotowe:", OUT / "data.yaml")